In [11]:
from pymatgen.core import Structure, Element
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

CIF_DIR = r'C:\Users\Sasha\repos\genetic_algo\new_ionic_cond_predictor\breed-2.0\cifs'
CLEAN_DIR = r'C:\Users\Sasha\repos\genetic_algo\new_ionic_cond_predictor\breed-2.0\cifs_cleaned'
ZEO_FEATURES = r'C:\Users\Sasha\repos\genetic_algo\new_ionic_cond_predictor\breed-2.0\zeo_features_combined.csv'

zeo = pd.read_csv(ZEO_FEATURES)
zeo['id'] = zeo['id'].astype(str)
succeeded = set(zeo.dropna(subset=['largest_included_sphere'])['id'].values)
all_cifs = [f.replace('.cif', '') for f in os.listdir(CIF_DIR) if f.endswith('.cif')]
failed_ids = set(all_cifs) - succeeded

os.makedirs(CLEAN_DIR, exist_ok=True)

np.random.seed(42)

cleaned = 0
errors = 0

for cif_id in sorted(failed_ids):
    cif_path = os.path.join(CIF_DIR, f'{cif_id}.cif')
    if not os.path.exists(cif_path):
        continue

    try:
        struct = Structure.from_file(cif_path)

        if struct.is_ordered:
            struct.to(filename=os.path.join(CLEAN_DIR, f'{cif_id}.cif'))
            cleaned += 1
            continue

        # Make supercell
        struct.make_supercell([2, 2, 2])

        new_species = []
        new_coords = []

        for site in struct:
            if site.is_ordered:
                new_species.append(str(site.specie))
                new_coords.append(site.frac_coords)
            else:
                species_occs = [(str(sp), occ) for sp, occ in site.species.items()]
                for sp_str, occ in sorted(species_occs, key=lambda x: -x[1]):
                    if np.random.random() < occ:
                        elem_name = ''
                        for char in sp_str:
                            if char.isalpha():
                                elem_name += char
                            else:
                                break
                        try:
                            Element(elem_name)
                            new_species.append(elem_name)
                            new_coords.append(site.frac_coords)
                            break
                        except:
                            continue

        if len(new_species) < 2:
            errors += 1
            continue

        clean = Structure(struct.lattice, new_species, new_coords)

        # Merge sites that are too close (< 1.0 angstrom)
        clean.merge_sites(tol=1.0, mode='delete')

        clean.to(filename=os.path.join(CLEAN_DIR, f'{cif_id}.cif'))
        cleaned += 1

    except Exception as e:
        print(f'ERROR {cif_id}: {e}')
        errors += 1

print(f'\nDone: {cleaned} cleaned, {errors} errors out of {len(failed_ids)}')


Done: 107 cleaned, 0 errors out of 107


In [13]:
from pymatgen.core import Structure
import os
import numpy as np

CIF_DIR = r'C:\Users\Sasha\repos\genetic_algo\new_ionic_cond_predictor\breed-2.0\cifs'

# Check a few ordered structures that fail Zeo++
ordered_failures = ['02p', '0iv', '11b', '47i', '4p7']

for cif_id in ordered_failures:
    cif_path = os.path.join(CIF_DIR, f'{cif_id}.cif')
    try:
        struct = Structure.from_file(cif_path)
        
        # Find minimum interatomic distance
        all_dists = struct.distance_matrix
        np.fill_diagonal(all_dists, 999)
        min_dist = all_dists.min()
        min_idx = np.unravel_index(all_dists.argmin(), all_dists.shape)
        
        site_a = struct[min_idx[0]]
        site_b = struct[min_idx[1]]
        
        print(f'{cif_id}: {len(struct)} sites, min distance = {min_dist:.3f} A')
        print(f'  Closest pair: {site_a.specie} -- {site_b.specie}')
        print(f'  Positions: {site_a.frac_coords.round(3)} -- {site_b.frac_coords.round(3)}')
        print()
    except Exception as e:
        print(f'{cif_id}: ERROR {e}\n')

02p: 50 sites, min distance = 2.029 A
  Closest pair: P -- S
  Positions: [0.993 0.965 0.502] -- [0.801 0.946 0.59 ]

0iv: 12 sites, min distance = 2.060 A
  Closest pair: P -- S
  Positions: [0.   0.   0.83] -- [0.67  0.    0.759]

11b: 8 sites, min distance = 2.467 A
  Closest pair: Li -- P
  Positions: [0.   0.   0.75] -- [0.667 0.333 0.75 ]

47i: 26 sites, min distance = 1.359 A
  Closest pair: P -- O
  Positions: [0.584 0.791 0.626] -- [0.662 0.724 0.479]

4p7: 5 sites, min distance = 1.954 A
  Closest pair: Li -- O
  Positions: [0.  0.5 0.5] -- [0.5 0.5 0.5]

